# LangChain Fundamentals — Building Blocks

This notebook walks through every core LangChain concept, from a basic LLM call to a fully functional agent.

**What you'll learn:**
1. Chat models — calling an LLM
2. Prompt templates — reusable, parameterized prompts
3. Output parsers — structured LLM output
4. LCEL chains — the `|` pipe syntax
5. Tools — making Python functions LLM-callable
6. Tavily web search — a real tool example
7. Function calling — LLM decides to call tools
8. Agents — the full ReAct loop

Run each cell in order. Read the markdown, then run the code.

## Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)
print(f"Loaded .env from: {env_path}")

---
## 1. Chat Models — Calling an LLM

LangChain provides a unified interface to many LLM providers through **Chat Models**.

| Class | Provider | Package |
|-------|----------|---------|
| `ChatOpenAI` | OpenAI (GPT-4o, etc.) | `langchain-openai` |
| `ChatAnthropic` | Anthropic (Claude) | `langchain-anthropic` |
| `ChatGoogleGenerativeAI` | Google (Gemini) | `langchain-google-genai` |

All of them share the same `.invoke()`, `.stream()`, `.batch()` interface.

**Key parameters:**
- `model` — which model to use (e.g. `"gpt-4o"`)
- `temperature` — 0 = deterministic, 1 = creative

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# The simplest possible call — a list of (role, content) tuples
response = llm.invoke([
    ("system", "You are a helpful assistant. Keep answers under 2 sentences."),
    ("user", "What is Python?"),
])

print("Content:", response.content)
print("Tokens:", response.usage_metadata)

### Message Types

LangChain has explicit message classes. The tuple syntax above is shorthand:

| Tuple | Class | Meaning |
|-------|-------|---------|
| `("system", "...")` | `SystemMessage` | Sets behavior/persona |
| `("user", "...")` | `HumanMessage` | User's input |
| `("assistant", "...")` | `AIMessage` | LLM's response |

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# Explicit message objects — equivalent to the tuples above
response = llm.invoke([
    SystemMessage(content="You are a pirate. Answer in pirate speak."),
    HumanMessage(content="What's the weather like?"),
])

print(response.content)

---
## 2. Prompt Templates — Reusable, Parameterized Prompts

`ChatPromptTemplate` lets you define prompts with `{variables}` that get filled in at runtime.

**Why not just use f-strings?**
- Templates are serializable (save/load, push to LangSmith)
- Templates integrate with LCEL chains (next section)
- Templates validate that required variables are provided

**Key methods:**
- `ChatPromptTemplate.from_messages([...])` — from a list of (role, template) tuples
- `.invoke({"var": "value"})` — fill in variables and return messages
- `.format_messages(...)` — same but returns message objects

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Answer concisely."),
    ("user", "{question}"),
])

# See what the template produces
messages = prompt.invoke({"domain": "astronomy", "question": "How far is the Moon?"})
print("Generated messages:")
for m in messages.messages:
    print(f"  [{m.type}] {m.content}")

# Pass directly to the LLM
response = llm.invoke(messages)
print(f"\nAnswer: {response.content}")

---
## 3. Output Parsers — Structured LLM Output

LLMs return `AIMessage` objects. Parsers extract the useful part.

| Parser | Output | Use case |
|--------|--------|---------|
| `StrOutputParser` | Plain string | Most common — just get the text |
| `JsonOutputParser` | Python dict | When you need structured data |
| `PydanticOutputParser` | Pydantic model | Type-safe structured output |

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Without parser: you get an AIMessage object
raw = llm.invoke([("user", "Say hello")])
print(f"Without parser: {type(raw).__name__} → {raw}")

# With parser: you get a clean string
parsed = parser.invoke(raw)
print(f"With parser:    {type(parsed).__name__} → {parsed}")

---
## 4. LCEL Chains — The `|` Pipe Syntax

**LCEL** (LangChain Expression Language) lets you chain components with `|`:

```python
chain = prompt | llm | parser
```

This creates a **pipeline** where:
1. `prompt` takes your variables → produces messages
2. `llm` takes messages → produces an AIMessage
3. `parser` takes AIMessage → produces a string

The chain itself is a `Runnable` with the same interface:
- `.invoke(input)` — run once
- `.stream(input)` — stream tokens as they arrive
- `.batch([input1, input2])` — run multiple inputs in parallel

In [ ]:
# Build a chain: prompt → LLM → string output
chain = prompt | llm | parser

# invoke() — single call
result = chain.invoke({"domain": "history", "question": "Who built the pyramids?"})
print(f"invoke(): {result}")

In [ ]:
# stream() — see tokens arrive in real time
print("stream(): ", end="")
for chunk in chain.stream({"domain": "science", "question": "What is DNA?"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
# batch() — multiple inputs in parallel
results = chain.batch([
    {"domain": "geography", "question": "Tallest mountain?"},
    {"domain": "music", "question": "Who composed the Moonlight Sonata?"},
])
for r in results:
    print(f"  → {r}")

### Multi-step chains — chaining multiple LLM calls with custom logic

A chain doesn't have to be `prompt | llm | parser`. You can chain **multiple LLM calls** with custom Python functions in between using `RunnableLambda`.

This is for **deterministic pipelines** — you always know the sequence of steps in advance:

```
Topic → [Search Tavily] → [LLM 1: Summarize] → [LLM 2: Write tweet] → Output
```

**Key building blocks:**
- `RunnableLambda(fn)` — wraps any Python function as a chainable step
- `RunnablePassthrough` — passes input through unchanged (useful for combining)
- `|` — chains them together just like prompt | llm | parser

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(max_results=2)

# Step 1: Search the web (a plain Python function wrapped as a Runnable)
def search_topic(inputs: dict) -> dict:
    response = tavily_search.invoke(inputs["topic"])
    raw_text = "\n".join(r["content"][:300] for r in response["results"])
    return {"topic": inputs["topic"], "research": raw_text}

# Step 2: LLM call #1 — summarize the research
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following research into 3 bullet points."),
    ("user", "Topic: {topic}\n\nResearch:\n{research}"),
])
summarize_chain = summarize_prompt | llm | parser

# Step 3: LLM call #2 — turn the summary into a tweet
tweet_prompt = ChatPromptTemplate.from_messages([
    ("system", "Turn this summary into a catchy tweet (max 280 chars). Include an emoji."),
    ("user", "{summary}"),
])
tweet_chain = tweet_prompt | llm | parser

# Wire it all together: search → summarize → tweet
full_pipeline = (
    RunnableLambda(search_topic)           # Python function: topic → research
    | RunnableLambda(lambda x: {           # reshape for the summarize prompt
        "topic": x["topic"],
        "research": x["research"],
    })
    | summarize_chain                       # LLM call #1: research → summary
    | RunnableLambda(lambda summary: {      # reshape for the tweet prompt
        "summary": summary,
    })
    | tweet_chain                           # LLM call #2: summary → tweet
)

# Run it
print("Pipeline: Search → Summarize (LLM #1) → Tweet (LLM #2)\n")
tweet = full_pipeline.invoke({"topic": "AI agents in software engineering"})
print(f"Tweet: {tweet}")

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Alternative pattern: RunnablePassthrough.assign() to accumulate intermediate results
# Each .assign() adds a new key, keeping all previous keys

# Step 1 & 2 combined: start with topic, assign research, then summary, then tweet
pipeline_v2 = (
    RunnablePassthrough.assign(
        research=lambda x: "\n".join(
            r["content"][:300]
            for r in tavily_search.invoke(x["topic"])["results"]
        )
    )
    | RunnablePassthrough.assign(
        summary=summarize_chain    # uses {topic} and {research} from accumulated state
    )
    | RunnablePassthrough.assign(
        tweet=tweet_chain           # uses {summary} from accumulated state
    )
)

# Run it — you get ALL intermediate results back
result = pipeline_v2.invoke({"topic": "quantum computing breakthroughs 2025"})

print("=== Full pipeline result (all intermediate values) ===\n")
print(f"Topic:   {result['topic']}\n")
print(f"Research (first 200 chars):\n  {result['research'][:200]}...\n")
print(f"Summary:\n  {result['summary']}\n")
print(f"Tweet:\n  {result['tweet']}")

### Chains recap — fixed, deterministic pipelines

Chains **always** follow a fixed sequence. Simple or multi-step, the LLM has no choice — it can't decide to skip a step, loop, or branch.

| Pattern | Example | Best for |
|---------|---------|----------|
| Simple chain | `prompt \| llm \| parser` | Q&A, translation, classification |
| Multi-step chain | `search → LLM #1 → LLM #2 → parse` | Pipelines where each stage is known in advance |
| Accumulating chain | `RunnablePassthrough.assign(...)` | When you need intermediate results |

**Key takeaway:** If you can draw the pipeline as a straight line on a whiteboard, a chain is the right abstraction.

But what if the LLM needs to **decide at runtime** whether to look something up or do a calculation? That's where tools and agents come in.

---
## 5. Tools — Making Python Functions LLM-Callable

The `@tool` decorator turns any Python function into something an LLM can call.

**What matters:**
- **Function name** — the LLM uses this to identify the tool
- **Docstring** — the LLM reads this to decide *when* to use the tool
- **Type hints** — the LLM uses these to know what arguments to pass

The LLM never runs your code directly. It outputs a structured JSON request saying *"call this tool with these arguments"*, and your code executes it.

In [ ]:
from datetime import datetime
from langchain_core.tools import tool

@tool
def get_current_time() -> str:
    """Return the current date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

@tool
def multiply(a: float, b: float) -> str:
    """Multiply two numbers together."""
    return str(a * b)

# You can call tools directly like normal functions
print("Direct call:")
print(f"  get_current_time(): {get_current_time.invoke({})}")
print(f"  multiply(7, 6):     {multiply.invoke({'a': 7, 'b': 6})}")

# What the LLM sees — the tool's schema
print(f"\nTool name:        {multiply.name}")
print(f"Tool description: {multiply.description}")
print(f"Tool args schema: {multiply.args_schema.model_json_schema()}")

---
## 6. Tavily Web Search — A Real Tool

[Tavily](https://tavily.com) is a search API designed for LLM applications. LangChain has a built-in integration.

**Response structure:** `TavilySearch.invoke(query)` returns a dict:
```python
{
    "query": "...",
    "results": [
        {"url": "...", "title": "...", "content": "...", "score": 0.9},
        ...
    ],
    "response_time": 0.8
}
```

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_results=2)
response = tavily.invoke("LangChain framework 2026")

print(f"Keys: {list(response.keys())}")
print(f"Results: {len(response['results'])}")
print()

for i, r in enumerate(response["results"], 1):
    print(f"Result {i}:")
    print(f"  Title:   {r['title']}")
    print(f"  URL:     {r['url']}")
    print(f"  Content: {r['content'][:150]}...")
    print()

In [ ]:
# Wrap Tavily as a proper @tool so an agent can use it
@tool
def search_web(query: str) -> str:
    """Search the web for current information on any topic."""
    response = tavily.invoke(query)
    return "\n".join(
        f"- {r['content'][:200]}" for r in response["results"]
    )

print(search_web.invoke("population of Japan 2026"))

---
## 7. Function Calling — The LLM Decides to Use Tools

With `.bind_tools()`, you tell the LLM what tools are available. The LLM then **generates structured tool call requests** instead of (or alongside) text.

This is the bridge between chains and agents:
- **Chain:** input → LLM → text output (always)
- **Function calling:** input → LLM → *either* text output *or* tool call request

The LLM doesn't execute the tool — it just says *"I want to call `multiply` with `a=7, b=6`"*. Your code runs the tool and feeds the result back.

In [ ]:
# Bind tools to the LLM
tools = [search_web, multiply, get_current_time]
llm_with_tools = llm.bind_tools(tools)

# Ask something that DOESN'T need tools
response = llm_with_tools.invoke([("user", "What is 2 + 2?")])
print("No tools needed:")
print(f"  Content:    {response.content}")
print(f"  Tool calls: {response.tool_calls}")

In [ ]:
# Ask something that DOES need tools
response = llm_with_tools.invoke([("user", "What time is it right now?")])
print("Tools needed:")
print(f"  Content:    {repr(response.content)}")
print(f"  Tool calls: {response.tool_calls}")
print()
print("The LLM is asking us to run get_current_time().")
print("It did NOT run the tool itself — it just generated a structured request.")

In [ ]:
# Ask something that needs search
response = llm_with_tools.invoke([("user", "What is the GDP of France?")])
print("Search tool call:")
for tc in response.tool_calls:
    print(f"  Tool: {tc['name']}")
    print(f"  Args: {tc['args']}")

### The gap: who executes the tool and feeds the result back?

With just `bind_tools`, the LLM can *request* tool calls but nobody is running them.

You could do it manually:
1. Call the LLM
2. Check if it returned tool calls
3. Execute the tools
4. Feed results back to the LLM
5. Repeat until done

But this loop is exactly what an **agent** automates.

---
## 8. Agents — The Full ReAct Loop

`create_agent()` from `langchain.agents` wires everything together:
- An LLM with tools bound
- A loop that: reasons → calls tools → observes results → repeats

This is the **ReAct** (Reasoning + Acting) pattern:

```
User Query
  → LLM thinks: "I need to search for this"
  → LLM calls: search_web("...")
  → Tool returns results
  → LLM thinks: "Now I need to calculate"
  → LLM calls: multiply(a=..., b=...)
  → Tool returns result  
  → LLM thinks: "I have everything I need"
  → Final answer
```

Under the hood, `create_agent` builds a LangGraph `StateGraph` with two nodes ("agent" and "tools") and a conditional edge. You'll build your own graph from scratch in Exercise 2.

In [ ]:
from langchain.agents import create_agent

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like '2 + 2' or '100 * 3.14'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

agent = create_agent(
    llm,
    tools=[search_web, calculate, get_current_time],
    system_prompt="You are a helpful research assistant. Use tools when needed.",
)

print(f"Agent type: {type(agent).__name__}")
print("(It's a compiled LangGraph — more on this in Exercise 2)")

In [ ]:
# Simple query — the agent calls search_web
result = agent.invoke({"messages": [("user", "What is the population of Brazil?")]})
print(result["messages"][-1].content)

In [ ]:
# Multi-step query — the agent chains tools together
result = agent.invoke({
    "messages": [("user", "What is the GDP of India? Calculate 15% of it.")]
})
print(result["messages"][-1].content)

In [ ]:
# Stream to see the ReAct loop step by step
print("Streaming the ReAct loop:\n")

for event in agent.stream(
    {"messages": [("user", "What time is it? Also search for the latest Python version.")]},
    stream_mode="values",
):
    if event.get("messages"):
        msg = event["messages"][-1]
        if msg.type == "ai" and hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  Tool call: {tc['name']}({tc['args']})")
        elif msg.type == "tool":
            print(f"  Tool result: {msg.content[:120]}...")
        elif msg.type == "ai" and msg.content:
            print(f"\n  Final answer: {msg.content[:500]}")

---
## Summary — The LangChain Building Blocks

```
Level 1: Basic LLM Call      →  llm.invoke([messages])
Level 2: Prompt Template      →  ChatPromptTemplate + variables
Level 3: Chain (LCEL)         →  prompt | llm | parser  (fixed pipeline)
Level 4: Tools                →  @tool decorator — functions the LLM can call
Level 5: Function Calling     →  llm.bind_tools() — LLM decides when to call tools
Level 6: Agent                →  create_agent() — automated ReAct loop
```

**Key insight:** A chain is a fixed pipeline. An agent is a loop where the LLM decides what to do next.

**Next up:** Open `starter.py` and build your own Research Assistant agent with web search, calculator, and Wikipedia tools.